[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C43_Data_Engineering_Course/04_quality_filtering/04_quality_filtering.ipynb)

# 04 · 大规模质量过滤（用 numpy/pandas/标准库从零实现 + 算账）

目标：把 **启发式规则 → 分类器 → 困惑度过滤 + 语言识别 + PII 检测** 从零实现，拼成一条**多阶段流水**，并算清两笔账：**各阶段留存率连乘** 与 **「便宜在前」省下的成本**。

路线：启发式规则 → 词袋逻辑回归分类器 → 困惑度(玩具 n-gram LM) → PII 正则 → 多阶段流水 + 留存率 → ✏️ 练习 → 📖 答案 → 🧪 FineWeb/Dolma 留存率胶囊。

> 心智模型：**过滤是个连乘的漏斗，编排顺序决定成本**。便宜且筛得狠的放最前，最贵的（困惑度）放最后只看少量数据。我们写过滤*逻辑*与*成本账*。

## 1 · 启发式规则（Gopher/C4 风格，最便宜的第一关）

每条规则就是一个统计量上的不等式：文档长度、符号比、重复行比、停用词比。
每篇 O(文档长度)、可并行，是漏斗最宽的一段。我们实现一组规则并验证它能筛掉明显垃圾。

In [ ]:
import numpy as np, re, hashlib
import pandas as pd
from collections import Counter
rng = np.random.default_rng(0)

STOPWORDS = {'the','is','of','and','to','a','in','that','it','for','on','with','as','are','be'}

def heuristic_pass(text):
    '''返回 True=保留, False=删除。Gopher/C4 风格规则的合取。'''
    words = re.findall(r'\w+', text.lower())
    n = len(words)
    if n < 5 or n > 100_000:
        return False                       # 文档过短/过长
    avg_word_len = np.mean([len(w) for w in words])
    if not (3.0 <= avg_word_len <= 10.0):
        return False                       # 平均词长异常 -> 乱码
    symbol_ratio = sum(text.count(ch) for ch in '#…') / max(len(text), 1)
    if symbol_ratio > 0.1:
        return False                       # 符号过多
    stop_ratio = sum(1 for w in words if w in STOPWORDS) / n
    if stop_ratio < 0.05:
        return False                       # 停用词过少 -> 关键词堆砌
    return True

good = 'The cat is sleeping on the warm mat and it is very happy today in the sun'
garbage1 = 'buy cheap pills viagra casino loans #### discount ######## sale'
garbage2 = 'a'                              # 太短
garbage3 = 'qwx zzz plkj mmm xkcd vbn'      # 无停用词、乱码
print('正常文本 ->', heuristic_pass(good))
print('SEO 垃圾 ->', heuristic_pass(garbage1))
print('过短     ->', heuristic_pass(garbage2))
print('乱码     ->', heuristic_pass(garbage3))
assert heuristic_pass(good) == True
assert heuristic_pass(garbage1) == False
assert heuristic_pass(garbage2) == False
assert heuristic_pass(garbage3) == False
print('✅ 启发式规则正确：保留正常文本、筛掉明显垃圾（每篇 O(长度)，最便宜）')

## 2 · 重复行比例（抓模板页/菜单堆砌）

网页常有大量重复的导航/菜单/页脚行。Gopher 用「重复行占比」筛掉它们。
实现一个 `repeated_line_fraction`，并验证模板页被识别为高重复。

In [ ]:
def repeated_line_fraction(text):
    '''重复行(出现>=2次的行)占总行数的比例。'''
    lines = [ln.strip() for ln in text.split('\n') if ln.strip()]
    if not lines:
        return 0.0
    counts = Counter(lines)
    repeated = sum(cnt for ln, cnt in counts.items() if cnt >= 2)
    return repeated / len(lines)

article = 'Introduction to systems.\nThis chapter explains design.\nWe begin with basics.'
template = 'Home\nAbout\nHome\nAbout\nHome\nAbout\nContact\nHome\nAbout'
print(f'正常文章 重复行比 = {repeated_line_fraction(article):.2f}')
print(f'模板页   重复行比 = {repeated_line_fraction(template):.2f}')
assert repeated_line_fraction(article) < 0.1, '正常文章重复行少'
assert repeated_line_fraction(template) > 0.5, '模板页重复行多'
# 把它接进启发式：重复行 > 0.3 删
def heuristic_pass_v2(text):
    return heuristic_pass(text) and repeated_line_fraction(text) <= 0.3
print('✅ 重复行规则正确：模板页/菜单堆砌被识别（接进启发式第一关）')

## 3 · 分类器过滤：词袋逻辑回归（从零训）

训一个轻量分类器区分「高质量参考」(正例) vs「随机网页」(负例)，给文档打质量分。
特征用 **哈希词袋**（hashed bag-of-words，fastText 的省钱思路），模型用 numpy 手写逻辑回归。

In [ ]:
DIM = 64                                   # 哈希特征维度

def featurize(text, dim=DIM):
    '''把文档哈希成 dim 维词袋特征（每个词哈希到一个桶，计数后归一化）。'''
    v = np.zeros(dim)
    words = re.findall(r'\w+', text.lower())
    for w in words:
        h = int(hashlib.md5(w.encode()).hexdigest(), 16) % dim
        v[h] += 1.0
    return v / max(np.linalg.norm(v), 1e-8)  # L2 归一化

def sigmoid(z): return 1.0 / (1.0 + np.exp(-z))

def train_logreg(X, y, epochs=300, lr=0.5):
    '''numpy 手写逻辑回归（梯度下降）。'''
    n, d = X.shape
    w = np.zeros(d); b = 0.0
    for _ in range(epochs):
        p = sigmoid(X @ w + b)
        gw = X.T @ (p - y) / n
        gb = np.mean(p - y)
        w -= lr * gw; b -= lr * gb
    return w, b

# 造正例(像维基的连贯文本) vs 负例(随机词堆)
wiki_words = 'the history of science shows that systematic study of nature leads to '\
             'understanding and knowledge through careful observation and experiment'.split()
def make_quality_doc():
    return ' '.join(rng.choice(wiki_words, size=30))
def make_random_doc():
    junk = [f'z{rng.integers(0,999)}' for _ in range(30)]
    return ' '.join(junk)

pos = [make_quality_doc() for _ in range(80)]
neg = [make_random_doc() for _ in range(80)]
X = np.array([featurize(d) for d in pos + neg])
y = np.array([1]*len(pos) + [0]*len(neg), dtype=float)
w, b = train_logreg(X, y)

def quality_score(text):
    return float(sigmoid(featurize(text) @ w + b))

s_good = quality_score('the careful study of nature leads to knowledge and understanding')
s_junk = quality_score('z12 z88 z3 z901 z44 z7 z200 z55')
print(f'高质量文档 质量分 = {s_good:.3f}')
print(f'随机词堆   质量分 = {s_junk:.3f}')
assert s_good > 0.5 > s_junk, '分类器应给高质量文档高分、随机文档低分'
# 训练集准确率
acc = np.mean((sigmoid(X @ w + b) >= 0.5) == y)
assert acc > 0.9, '训练集准确率应 > 90%'
print(f'✅ 分类器训练完成，训练集准确率 {acc:.0%}（轻量、可批量、坐镇管线中段）')

## 4 · 困惑度过滤：玩具 bigram LM（最贵，放最后）

在干净文本上统计 bigram 概率作为「品味标准」，给文档算困惑度 `PPL = exp(-mean log p)`。
**通顺文本困惑度低、乱序文本困惑度高**。这一关要跑 LM，最贵，所以放管线末端只看少量数据。

In [ ]:
def train_bigram_lm(corpus, vocab_smoothing=1.0):
    '''统计 bigram 计数，返回 (bigram_counts, unigram_counts, vocab)。'''
    bigram = Counter(); unigram = Counter(); vocab = set()
    for text in corpus:
        toks = ['<s>'] + re.findall(r'\w+', text.lower()) + ['</s>']
        vocab.update(toks)
        for a, b_ in zip(toks[:-1], toks[1:]):
            bigram[(a, b_)] += 1; unigram[a] += 1
    return bigram, unigram, vocab

def perplexity(text, bigram, unigram, vocab):
    V = len(vocab)
    toks = ['<s>'] + re.findall(r'\w+', text.lower()) + ['</s>']
    logp = 0.0; N = 0
    for a, b_ in zip(toks[:-1], toks[1:]):
        # 加一平滑的 bigram 概率
        prob = (bigram[(a, b_)] + 1.0) / (unigram[a] + V)
        logp += np.log(prob); N += 1
    return float(np.exp(-logp / max(N, 1)))

# 干净语料（重复几遍让 bigram 统计更稳，模拟「在大量干净文本上训」）
clean_corpus = ['the cat sat on the mat',
                'the dog ran in the park',
                'a cat and a dog sat on the mat in the park',
                'the sun is warm and the sky is clear today'] * 5
bg, ug, vocab = train_bigram_lm(clean_corpus)

fluent = 'the cat sat on the mat'
scrambled = 'mat the on sat cat the'              # 同样的词，乱序 -> 破坏 bigram
gibberish = 'zzz qqq xxx www vvv'                 # 全是没见过的词
ppl_f = perplexity(fluent, bg, ug, vocab)
ppl_s = perplexity(scrambled, bg, ug, vocab)
ppl_g = perplexity(gibberish, bg, ug, vocab)
print(f'通顺句 PPL = {ppl_f:6.1f}  <- 低（模型不意外）')
print(f'乱序句 PPL = {ppl_s:6.1f}  <- 高（bigram 被打乱）')
print(f'乱码   PPL = {ppl_g:6.1f}  <- 高（全是 OOV）')
# 稳健的结论：通顺文本困惑度最低（最「像」自然语言）。
# 乱序 vs 乱码 谁更高取决于平滑细节，不强求；关键是「通顺」远低于两者。
assert ppl_f < ppl_s, '通顺应远低于乱序'
assert ppl_f < ppl_g, '通顺应远低于乱码'
assert ppl_f < 10 and ppl_s > 15 and ppl_g > 15, '通顺低、不通顺高'
print('✅ 困惑度过滤正确：越像自然语言 PPL 越低；阈值卡掉高 PPL（但最贵，放最后！）')

## 5 · PII 检测：正则脱敏（合规横切，几乎不删只改）

邮箱/电话/密钥有固定格式，正则又快又准。PII 检测通常**脱敏而非删除**，所以留存≈100%。
实现一个正则脱敏器，并验证它抓住结构化 PII。

In [ ]:
PII_PATTERNS = {
    'EMAIL': re.compile(r'[\w.+-]+@[\w-]+\.[\w.-]+'),
    'PHONE': re.compile(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b'),
    'KEY':   re.compile(r'\b[A-Za-z0-9]{32,}\b'),       # 长 token 像 API key
}

def redact_pii(text):
    '''返回 (脱敏后文本, 命中计数 dict)。'''
    hits = {}
    out = text
    for name, pat in PII_PATTERNS.items():
        found = pat.findall(out)
        hits[name] = len(found)
        out = pat.sub(f'[{name}]', out)
    return out, hits

doc = 'Contact me at john.doe@example.com or 555-123-4567. Key: ' + 'a'*40
clean, hits = redact_pii(doc)
print('原文 :', doc[:60], '...')
print('脱敏 :', clean[:60], '...')
print('命中 :', hits)
assert hits['EMAIL'] == 1 and hits['PHONE'] == 1 and hits['KEY'] == 1
assert '@' not in clean and '[EMAIL]' in clean
assert '555-123-4567' not in clean
# PII 是脱敏不是删除：文档仍然保留
print('✅ PII 正则脱敏正确：抓住结构化 PII，脱敏而非删除（留存≈100%，但仍要扫描成本）')

## 6 · 多阶段流水：留存率连乘 + 「便宜在前」的成本账

把前面的过滤器拼成一条流水，用 pandas 跟踪每阶段留存率，并对比**最优顺序 vs 最坏顺序**的成本。
这是本模块的核心：**同样的过滤效果，编排顺序决定成本差一个数量级**。

In [ ]:
# 造一个混合语料：高质量 + SEO垃圾 + 乱码 + 模板页
def make_corpus(n=400):
    docs = []
    for _ in range(n):
        r = rng.random()
        if r < 0.35:
            docs.append(' '.join(rng.choice(wiki_words, size=rng.integers(20,40))))  # 高质量
        elif r < 0.6:
            docs.append('buy cheap ' + ' '.join([f'#deal{i}' for i in range(15)]))    # SEO垃圾
        elif r < 0.8:
            docs.append(' '.join([f'z{rng.integers(0,99)}' for _ in range(25)]))      # 乱码
        else:
            docs.append('\n'.join(['Home','About','Home','About']*4))                 # 模板页
    return docs

corpus = make_corpus(400)

# 各阶段：(名字, 单篇成本(相对), 过滤函数 doc->bool)
STAGES = [
    ('heuristic',  1.0,   lambda d: heuristic_pass_v2(d)),
    ('classifier', 10.0,  lambda d: quality_score(d) >= 0.5),
    ('perplexity', 1000.0, lambda d: perplexity(d, bg, ug, vocab) < 5000),
]

def run_pipeline(docs, stages):
    '''按给定阶段顺序跑，返回 (留存的docs, 每阶段记录 DataFrame, 总成本)。'''
    survivors = list(docs)
    rows = []; total_cost = 0.0
    for name, cost, fn in stages:
        n_in = len(survivors)
        total_cost += cost * n_in              # 该阶段处理 n_in 篇
        survivors = [d for d in survivors if fn(d)]
        n_out = len(survivors)
        rows.append({'stage': name, 'in': n_in, 'out': n_out,
                     'retention': n_out/max(n_in,1), 'cost_here': cost*n_in})
    return survivors, pd.DataFrame(rows), total_cost

surv, df, cost_optimal = run_pipeline(corpus, STAGES)        # 便宜在前
print(df.to_string(index=False))
end_to_end = len(surv) / len(corpus)
print(f'\n端到端留存 = {end_to_end:.1%} (= 各阶段连乘)，总成本 = {cost_optimal:,.0f}')

# 最坏顺序：困惑度放最前
_, _, cost_worst = run_pipeline(corpus, STAGES[::-1])
print(f'最坏顺序(困惑度在前) 总成本 = {cost_worst:,.0f}')
print(f'便宜在前 省下 {cost_worst/cost_optimal:.1f}x 成本')
assert cost_optimal < cost_worst, '便宜在前应比最坏顺序省'
assert abs(end_to_end - df['retention'].prod()) < 1e-9, '端到端留存=各阶段连乘'
print('✅ 多阶段流水正确：留存率连乘成漏斗；编排顺序决定成本')

---
## ✏️ 练习 1：语言识别代理（按字符特征分流）

实现一个简易 `detect_language(text)`：用**字符 n-gram 特征**区分英语 vs 非英语。
提示：英语文本 ASCII 字母占比高。返回 `'en'`（ASCII 字母占比 ≥ 0.6）或 `'other'`。

（真实用 fastText langid；这里用 ASCII 占比做代理，体会 LID 便宜且放管线最前。）

In [ ]:
def detect_language(text):
    # TODO: 算 ASCII 字母(a-zA-Z)占所有非空白字符的比例；
    #       >= 0.6 返回 'en'，否则 'other'。空文本返回 'other'。
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert detect_language('the quick brown fox jumps over') == 'en'
assert detect_language('这是一段中文文本没有空格分词') == 'other'
assert detect_language('') == 'other'
# 混合：大部分中文 -> other
assert detect_language('你好 hello 世界 world 测试 文本 数据') == 'other'
print('✅ 练习 1 通过：语言识别代理正确（便宜，放管线最前先按语言分流）')

## ✏️ 练习 2：困惑度阈值的留存率调参

给定一批文档的困惑度数组 `ppls` 和目标留存率 `target_keep`，
实现 `ppl_threshold(ppls, target_keep)`：找一个阈值，使**保留(PPL ≤ 阈值)的比例 ≈ target_keep**。

提示：保留低困惑度的 = 取分位数。target_keep=0.8 → 阈值 = 80% 分位数。

In [ ]:
def ppl_threshold(ppls, target_keep):
    # TODO: 返回 ppls 的 target_keep 分位数（np.quantile），
    #       使 PPL <= 该阈值 的比例约为 target_keep
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
ppls = np.array([10, 20, 30, 40, 50, 60, 70, 80, 90, 100], dtype=float)
thr = ppl_threshold(ppls, 0.8)
keep_frac = np.mean(ppls <= thr)
assert abs(keep_frac - 0.8) <= 0.1, f'保留比例应≈0.8, 实际{keep_frac}'
# 目标留存越高，阈值越大（留得越多）
assert ppl_threshold(ppls, 0.9) >= ppl_threshold(ppls, 0.5)
print(f'目标留存 0.8 -> PPL 阈值 = {thr:.0f}, 实际留存 {keep_frac:.0%}')
print('✅ 练习 2 通过：能据目标留存率反解困惑度阈值')

## ✏️ 练习 3：扩展 PII 正则（信用卡号）

给第 5 节的 PII 检测加一个 **信用卡号** 模式：16 位数字，可有空格/连字符分组
（如 `4111 1111 1111 1111` 或 `4111-1111-1111-1111` 或 `4111111111111111`）。

实现 `redact_with_cc(text)`：在原有 PII 基础上额外脱敏信用卡号为 `[CC]`，返回 `(脱敏文本, hits)`。

In [ ]:
def redact_with_cc(text):
    # TODO: 先定义信用卡正则（16位数字，4组，分隔符为空格/连字符/无）；
    #       先脱敏 CC（避免被 KEY 规则误吞），再调 redact_pii 处理其余；
    #       合并 hits（含 'CC' 键）返回。
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
doc = 'Card 4111 1111 1111 1111 and email a@b.com'
clean, hits = redact_with_cc(doc)
assert hits['CC'] == 1, '应抓到 1 个信用卡号'
assert hits['EMAIL'] == 1, '原有 EMAIL 规则仍生效'
assert '[CC]' in clean and '4111' not in clean
# 连字符格式也要抓
_, hits2 = redact_with_cc('pay 4111-1111-1111-1111 now')
assert hits2['CC'] == 1
print('✅ 练习 3 通过：PII 正则可扩展，信用卡号被脱敏')

## ✏️ 练习 4：最优阶段排序（成本 × 留存率）

给定各阶段 `(名字, 单篇成本, 留存率)`，实现 `optimal_order(stages)`：
返回使**总成本最小**的阶段顺序。

提示：贪心——总成本 = Σ cost_i × ∏_{j 在 i 前} keep_j。按 `cost / (1 - retention)`（性价比：单位成本能淘汰多少）升序排，便宜且淘汰高的在前。

In [ ]:
def optimal_order(stages):
    '''stages: list of (name, cost, retention). 返回重排后的 list。'''
    # TODO: 按 cost/(1-retention+1e-9) 升序排序（性价比高=单位成本淘汰多 的放前面）
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
def total_cost_of_order(stages, n=1000):
    cost = 0.0; remaining = n
    for name, c_, k_ in stages:
        cost += c_ * remaining; remaining *= k_
    return cost

stages = [('ppl', 1000.0, 0.8), ('heur', 1.0, 0.4), ('clf', 10.0, 0.5)]
ordered = optimal_order(stages)
names = [s[0] for s in ordered]
# 便宜且淘汰高的 heur 应在最前，贵的 ppl 应在最后
assert names[0] == 'heur', 'heuristic(便宜+淘汰高)应在最前'
assert names[-1] == 'ppl', 'perplexity(最贵)应在最后'
# 最优顺序成本应 <= 原顺序
assert total_cost_of_order(ordered) <= total_cost_of_order(stages)
print('最优顺序:', names)
print(f'最优成本 {total_cost_of_order(ordered):,.0f} vs 原序 {total_cost_of_order(stages):,.0f}')
print('✅ 练习 4 通过：按性价比排序，便宜且筛得狠的在前 -> 总成本最小')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def detect_language(text):
    chars = [ch for ch in text if not ch.isspace()]
    if not chars:
        return 'other'
    ascii_alpha = sum(1 for ch in chars if ch.isascii() and ch.isalpha())
    return 'en' if ascii_alpha / len(chars) >= 0.6 else 'other'

In [ ]:
# 练习 2 参考答案
def ppl_threshold(ppls, target_keep):
    return float(np.quantile(ppls, target_keep))

In [ ]:
# 练习 3 参考答案
def redact_with_cc(text):
    cc_pat = re.compile(r'\b(?:\d{4}[ -]?){3}\d{4}\b')
    n_cc = len(cc_pat.findall(text))
    out = cc_pat.sub('[CC]', text)
    out, hits = redact_pii(out)
    hits['CC'] = n_cc
    return out, hits

In [ ]:
# 练习 4 参考答案
def optimal_order(stages):
    return sorted(stages, key=lambda s: s[1] / (1.0 - s[2] + 1e-9))

---
## 🧪 真实数据胶囊：FineWeb / Dolma 留存率与原始数据账

用 FineWeb/Dolma/CommonCrawl 的公开留存率量级，算两笔真实的账：
① 端到端留存率连乘 → 原始要囤多少数据才够训练目标；② 「便宜在前」相对「困惑度在前」省下多少昂贵计算。

（带 try/except：本环境不联网，直接用内置的真实量级数字。）

In [ ]:
# FineWeb/Dolma/CCNet 公开的各阶段留存率量级（约数）
STAGE_RETENTION = {
    'language_id':  0.45,    # 英语项目，删掉多数非英语
    'url_filter':   0.85,    # 黑名单域名
    'heuristic':    0.50,    # Gopher/C4 规则
    'dedup':        0.50,    # 去重(模块01)，量级
    'classifier':   0.40,    # 质量分类器
    'perplexity':   0.80,    # 困惑度过滤
}
STAGE_COST = {  # 单篇相对成本（量级）
    'language_id': 1, 'url_filter': 0.1, 'heuristic': 1,
    'dedup': 5, 'classifier': 10, 'perplexity': 1000,
}

# ① 端到端留存 -> 原始数据需求
end_to_end = 1.0
for k in STAGE_RETENTION.values():
    end_to_end *= k
TARGET_TOKENS = 1.5e13                       # 想要的训练 token
raw_needed = TARGET_TOKENS / end_to_end
print(f'① 端到端留存 = {end_to_end:.1%}')
print(f'   要 {TARGET_TOKENS:.1e} 训练 token -> 需从 {raw_needed:.1e} 原始 token 出发')
print(f'   即原始数据要比训练数据多囤 {1/end_to_end:.0f}x（要好几个 CommonCrawl 快照）')

# ② 便宜在前 vs 困惑度在前 的成本
N = 1e12
def pipeline_cost(order):
    cost = 0.0; remaining = N
    for name in order:
        cost += STAGE_COST[name] * remaining
        remaining *= STAGE_RETENTION[name]
    return cost
cheap_first = ['url_filter','language_id','heuristic','dedup','classifier','perplexity']
ppl_first   = ['perplexity','classifier','dedup','heuristic','language_id','url_filter']
c_opt, c_bad = pipeline_cost(cheap_first), pipeline_cost(ppl_first)
print(f'\n② 便宜在前 总成本 = {c_opt:.2e}')
print(f'   困惑度在前 总成本 = {c_bad:.2e}')
print(f'   便宜在前省下 {c_bad/c_opt:.0f}x')

assert end_to_end < 0.1, 'CommonCrawl 端到端留存应是个位数百分比'
assert c_opt < c_bad, '便宜在前必须更省'
assert raw_needed > TARGET_TOKENS * 10, '原始数据需求远大于训练目标'
print('\n账目结论：过滤是连乘漏斗(留存个位数%)，原始要多囤十几倍；')
print('且「便宜在前」的编排让最贵阶段只看漏斗末端，省下数量级的成本 —— 这就是大规模过滤的工程核心。')

**🧪 胶囊练习**：实现 `expensive_stage_workload(order, retention, n=1e12)`：给定阶段顺序与各阶段留存率，返回**最贵阶段（perplexity）实际要处理的文档数**（= 它前面所有阶段筛剩的量）。用它证明：把 perplexity 放最后，它的工作量最小。

In [ ]:
def expensive_stage_workload(order, retention, n=1e12):
    # TODO: 沿 order 累乘留存率，返回 perplexity 阶段之前筛剩的文档数
    #       （即到达 perplexity 时的 remaining）
    raise NotImplementedError

In [ ]:
# 自测
ret = STAGE_RETENTION
last = expensive_stage_workload(cheap_first, ret)   # ppl 放最后
first = expensive_stage_workload(ppl_first, ret)    # ppl 放最前
assert first == 1e12, 'ppl 放最前时要处理全部 n 篇'
assert last < first, 'ppl 放最后时处理的文档数远少'
assert last < 1e12 * 0.1, 'ppl 放最后只看漏斗末端(<10%)'
print(f'perplexity 放最后只处理 {last:.2e} 篇；放最前要处理 {first:.2e} 篇')
print(f'放最后 vs 放最前：工作量降到 {last/first:.1%}')
print('✅ 胶囊练习通过：最贵阶段放最后，工作量最小')

In [ ]:
# 📖 胶囊参考答案
def expensive_stage_workload(order, retention, n=1e12):
    remaining = n
    for name in order:
        if name == 'perplexity':
            return remaining
        remaining *= retention[name]
    return remaining

---
## 🔧 旁注：真实管线里的质量过滤长什么样

本课的小模拟，在 FineWeb/Dolma 等真实管线里对应：

- **多阶段 DAG** 跑在 Spark/Ray 上，每阶段是一个 map 算子（逐文档独立、易并行），按「便宜在前」串成漏斗，每阶段的留存率与吞吐实时监控。
- **启发式** = Gopher/MassiveText 规则集（Rae 2021）、C4 规则（Raffel 2020）；**分类器** = fastText（Joulin 2017）训在「维基/书籍 vs 随机网页」；**困惑度** = CCNet 的 KenLM（Wenzek 2020）。
- **PII/语言** 是横切关注点：LID 放最前按语言分流，PII 用正则脱敏（Dolma 的 PII 工具）。

你在 numpy/pandas 里验证过的过滤逻辑与「便宜在前」的成本账，可几乎一对一搬到 Spark/Ray 的多阶段 pipeline 上。**判据本身（什么算好数据）属于 C21 的数据科学；本课负责把它工程化、算清留存率与成本。**

### 小结
- 质量过滤是**工程编排问题**：过滤器成本横跨 6 个数量级，顺序决定总成本。
- **多阶段漏斗**：便宜在前（语言识别/启发式）→ 中段分类器 → 最贵的困惑度放最后只看少量数据。
- 各阶段**留存率连乘** = 端到端留存（常个位数%），决定原始要囤多少数据。
- **横切**：语言识别(便宜+高淘汰,放最前)、PII(脱敏不删,留存≈100%)。
- 方法论：每个过滤器都从零实现+验证，每条管线都算**留存率账 + 成本账**。

下一站：**模块 05 · 溯源与去污染** —— 过滤后的数据，怎么证明它没泄漏评测题、怎么让它可追溯可复现？